# *Staphylococcus aureus* WGS Variant-Calling Pipeline

## Portfolio version

This notebook documents one selected paired-end *S. aureus* WGS run and the variant-calling workflow used for it.

**Workflow:** SRA/ENA reads → Trimmomatic → BWA-MEM → SAMtools → FreeBayes → VCF

**Run accession:** SRR8359173  
**Reference assembly:** GCF_000013425.1_ASM1342v1  
**Reference file:** `usa300_reference.fna`

> **Important interpretation:** the reported 58,548 records are raw FreeBayes variant calls under the parameters shown below. They are not presented as 58,548 experimentally validated mutations.

## 1. Download the paired-end reads

The reads were retrieved from the European Nucleotide Archive FTP path corresponding to SRA run **SRR8359173**.

In [ ]:
!wget -O final_1.fastq.gz "https://ftp.sra.ebi.ac.uk/vol1/fastq/SRR835/003/SRR8359173/SRR8359173_1.fastq.gz"
!wget -O final_2.fastq.gz "https://ftp.sra.ebi.ac.uk/vol1/fastq/SRR835/003/SRR8359173/SRR8359173_2.fastq.gz"

## 2. Reference genome

The analysis used the *S. aureus* USA300 reference assembly **GCF_000013425.1_ASM1342v1**.

In [ ]:
!wget -O usa300_reference.fna.gz "https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/013/425/GCF_000013425.1_ASM1342v1/GCF_000013425.1_ASM1342v1_genomic.fna.gz"
!gunzip -f usa300_reference.fna.gz
!mv GCF_000013425.1_ASM1342v1_genomic.fna usa300_reference.fna
!bwa index usa300_reference.fna

## 3. Read trimming with Trimmomatic

Paired-end reads were trimmed with:

- `LEADING:3` — remove low-quality bases at the start below Q3.
- `TRAILING:3` — remove low-quality bases at the end below Q3.
- `SLIDINGWINDOW:4:15` — scan four-base windows and trim when the window's average quality falls below Q15.
- `MINLEN:36` — discard reads shorter than 36 bases after trimming.

The paired outputs were used for alignment; unpaired reads were retained but not used in the paired-end alignment step.

In [ ]:
!TrimmomaticPE final_1.fastq.gz final_2.fastq.gz \
    final_1_paired.fastq.gz final_1_unpaired.fastq.gz \
    final_2_paired.fastq.gz final_2_unpaired.fastq.gz \
    LEADING:3 TRAILING:3 SLIDINGWINDOW:4:15 MINLEN:36

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
TrimmomaticPE: Started with arguments:
 final_1.fastq.gz final_2.fastq.gz final_1_paired.fastq.gz final_1_unpaired.fastq.gz final_2_paired.fastq.gz final_2_unpaired.fastq.gz LEADING:3 TRAILING:3 SLIDINGWINDOW:4:15 MINLEN:36
Multiple cores found: Using 2 threads
Quality encoding detected as phred33
Input Read Pairs: 5019157 Both Surviving: 4834525 (96.32%) Forward Only Surviving: 143389 (2.86%) Reverse Only Surviving: 27236 (0.54%) Dropped: 14007 (0.28%)
TrimmomaticPE: Completed successfully


### Trimming result

The recorded run contained **5,019,157 input read pairs**.

- Both mates survived: **4,834,525 pairs (96.32%)**
- Forward-only surviving reads: **143,389 (2.86%)**
- Reverse-only surviving reads: **27,236 (0.54%)**
- Dropped: **14,007 (0.28%)**

These numbers describe the trimming step; they do not by themselves establish that the reads are biologically correct.

## 4. Alignment with BWA-MEM

BWA-MEM was used to align the surviving paired reads to the selected reference genome.

The resulting SAM file contains read-to-reference alignments, including mapping positions, mapping quality and CIGAR strings.

In [ ]:
!bwa mem usa300_reference.fna final_1_paired.fastq.gz final_2_paired.fastq.gz > final_aligned.sam

## 5. Convert, sort and index the alignment

SAM is converted to BAM because BAM is a compact binary representation commonly used by downstream tools.

The BAM is coordinate-sorted and indexed so that downstream programs can efficiently access reads by genomic position.

In [ ]:
!samtools view -bS final_aligned.sam | samtools sort -o final_sorted.bam
!samtools index final_sorted.bam
!samtools flagstat final_sorted.bam

9669762 + 0 in total (QC-passed reads + QC-failed reads)
9669050 + 0 primary
0 + 0 secondary
712 + 0 supplementary
0 + 0 duplicates
0 + 0 primary duplicates
411625 + 0 mapped (4.26% : N/A)
410913 + 0 primary mapped (4.25% : N/A)
9669050 + 0 paired in sequencing
4834525 + 0 read1
4834525 + 0 read2
348224 + 0 properly paired (3.60% : N/A)
350558 + 0 with itself and mate mapped
60355 + 0 singletons (0.62% : N/A)
0 + 0 with mate mapped to a different chr
0 + 0 with mate mapped to a different chr (mapQ>=5)


## 6. Alignment QC: important limitation

The recorded alignment statistics show:

- Total reads: **9,669,762**
- Mapped reads: **411,625 (4.26%)**
- Properly paired: **348,224 (3.60%)**
- Marked duplicates: **0**

The low mapping percentage is an important limitation. It means most reads did not align to the selected reference, so the downstream variant calls should be treated as **preliminary/raw calls**, not as a validated set of biological mutations.

No dedicated duplicate-marking step was included in this version of the pipeline, so `0 duplicates` means no duplicates were marked—not that PCR duplication was proven absent.

A production analysis should investigate the sample/reference match, sequencing metadata and contamination/quality before making biological conclusions.

## 7. Variant calling with FreeBayes

FreeBayes was run on the sorted BAM against the reference.

Parameters used:

- `--min-coverage 1` — minimum total read coverage considered at a position.
- `--min-alternate-count 1` — at least one read supporting the alternate allele.
- `--min-alternate-fraction 0.01` — alternate allele fraction of at least 1%.

These are permissive settings. Therefore, the resulting count is a **raw call set** that requires downstream filtering and validation.

In [ ]:
!freebayes -f usa300_reference.fna final_sorted.bam \
    --min-coverage 1 \
    --min-alternate-count 1 \
    --min-alternate-fraction 0.01 > final_variants.vcf

!grep -v "^#" final_variants.vcf | wc -l

58548


## 8. Result

The final VCF contained **58,548 non-header variant records**.

This number should be reported as **58,548 raw FreeBayes calls**, not as 58,548 confirmed mutations.

### Recommended next steps for a production analysis

1. Verify the biological identity and metadata of the SRA sample.
2. Confirm that the reference is appropriate for that sample.
3. Perform comprehensive read-level QC.
4. Review mapping quality and coverage across the reference.
5. Mark/assess PCR duplicates where appropriate.
6. Apply stricter variant-quality, depth and allele-support filters.
7. Annotate the filtered variants before biological interpretation.

The purpose of this portfolio project is to demonstrate implementation and understanding of a WGS variant-calling workflow.

## 9. Interview summary

**One-sentence description:**

> I implemented a paired-end *S. aureus* WGS variant-calling workflow in a command-line/Colab environment, covering read trimming, reference alignment, BAM processing and FreeBayes variant calling, and I learned to interpret the resulting QC metrics and limitations rather than treating every raw call as a validated mutation.

**Tools:** Trimmomatic, BWA-MEM, SAMtools, FreeBayes, SRA/ENA, Linux shell.